In [ ]:
# =================================================================
# CELL 1 — setup and CSV picker
# =================================================================
%matplotlib widget
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import curve_fit
import glob
import ipywidgets as widgets
from IPython.display import display
 
 
def bi_gaussian(phi, mu1, sigma1, mu2, sigma2, p):
    g1 = np.exp(-0.5 * ((phi - mu1) / sigma1) ** 2) / (sigma1 * np.sqrt(2 * np.pi))
    g2 = np.exp(-0.5 * ((phi - mu2) / sigma2) ** 2) / (sigma2 * np.sqrt(2 * np.pi))
    return p * g1 + (1.0 - p) * g2
 
 
csv_files = sorted(glob.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError("No .csv files found in the current folder.")
 
csv_picker = widgets.Dropdown(
    options=csv_files,
    description='CSV file:',
    style={'description_width': 'initial'},
)
display(csv_picker)
 
 

In [ ]:
# =================================================================
# CELL 2 — load data using the file selected above
# =================================================================
DATASET_LABEL = csv_picker.value.rsplit('.', 1)[0]
print(f"Loading: {csv_picker.value}")
 
_raw = pd.read_csv(csv_picker.value, skiprows=1)
_raw = _raw.rename(columns={'Phi': 'phi', 'Mass (wt.%)': 'mass'})
_data = _raw[['phi', 'mass']].dropna(subset=['phi']).copy()
_data['mass'] = _data['mass'].fillna(0.0).astype(float)
_data['phi']  = _data['phi'].astype(float)
phi_labels = _data['phi'].to_numpy() - 0.5
mass_pct   = _data['mass'].to_numpy()
DENSITY = 2500.0  # kg/m3
 
mass_frac = mass_pct / mass_pct.sum()
 
print(f"Loaded {DATASET_LABEL}: {len(phi_labels)} points, "
      f"Phi range [{phi_labels.min():.2f}, {phi_labels.max():.2f}]")
 
 

In [ ]:
# =================================================================
# CELL 3 — interactive picker
# =================================================================
picked_points = []  # reset each time this cell runs
MAX_CLICKS = 2

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(phi_labels, mass_pct, width=0.90, color='#c0392b', alpha=0.55,
       edgecolor='grey', linewidth=0.5, label='Data')
ax.set_xlabel('Φ', fontsize=11)
ax.set_ylabel('Mass (wt.%)', fontsize=11)
ax.set_title('Click the peak of each mode (order doesn\'t matter)')
ax.set_xlim(phi_labels.min() - 1, phi_labels.max() + 1)
ax.set_ylim(bottom=0)
ax.grid(alpha=0.3)
ax.legend(fontsize=8)

colors = ['#2980b9', '#27ae60']


def onclick(event):
    if event.inaxes != ax or len(picked_points) >= MAX_CLICKS:
        return
    phi_click, y_click = event.xdata, event.ydata
    picked_points.append((phi_click, y_click))

    i = len(picked_points) - 1
    ax.axvline(phi_click, color=colors[i], linestyle='--', lw=1.5)
    ax.annotate(f'peak {i+1}\nΦ={phi_click:.2f}',
                xy=(phi_click, y_click), xytext=(5, 5),
                textcoords='offset points', color=colors[i], fontsize=9)
    fig.canvas.draw_idle()

    print(f"Peak {i+1} picked: Phi={phi_click:.3f}, wt%={y_click:.3f}")
    if len(picked_points) == MAX_CLICKS:
        print("Both peaks picked - run the next cell to fit.")


cid = fig.canvas.mpl_connect('button_press_event', onclick)
plt.tight_layout()
plt.show()

In [ ]:
# =================================================================
# CELL 4 — build p0 from clicks and fit
# Only run this AFTER both clicks are registered in Cell 2's output.
# =================================================================
if len(picked_points) < 2:
    raise RuntimeError(
        "Only got {} click(s) so far. Click twice on the Cell 2 plot "
        "(one per mode peak) before running this cell."
        .format(len(picked_points))
    )

# Sort by phi so pop 1 = coarse (smaller phi), pop 2 = fine (larger phi)
(phi1, y1), (phi2, y2) = sorted(picked_points, key=lambda pt: pt[0])

mu1_guess = phi1
mu2_guess = phi2

# Rough sigma guess: 
default_sigma = 1.0
y1_frac = max(y1 / mass_pct.sum() if mass_pct.sum() > 0 else 0, 1e-6)
y2_frac = max(y2 / mass_pct.sum() if mass_pct.sum() > 0 else 0, 1e-6)

w_guess = y1 / (y1 + y2) if (y1 + y2) > 0 else 0.5

# peak height of a weighted Gaussian component ~= weight / (sigma*sqrt(2pi))
# => sigma ~= weight / (peak_height_frac * sqrt(2pi))
sigma1_guess = max(w_guess / (y1_frac * np.sqrt(2 * np.pi)), 0.3) if y1_frac > 0 else default_sigma
sigma2_guess = max((1 - w_guess) / (y2_frac * np.sqrt(2 * np.pi)), 0.3) if y2_frac > 0 else default_sigma

# clip to something reasonable so a noisy click doesn't blow this up
sigma1_guess = float(np.clip(sigma1_guess, 0.3, 6.0))
sigma2_guess = float(np.clip(sigma2_guess, 0.3, 6.0))

p0_g = [mu1_guess, sigma1_guess, mu2_guess, sigma2_guess, w_guess]
lb_g = [-12., 0.1, -12., 0.1, 0.01]
ub_g = [ 14., 8.0,  14., 8.0, 0.99]
p0_g = list(np.clip(p0_g, lb_g, ub_g))

print(f"Initial guess from clicks: mu1={p0_g[0]:.3f}, sigma1={p0_g[1]:.3f}, "
      f"mu2={p0_g[2]:.3f}, sigma2={p0_g[3]:.3f}, p={p0_g[4]:.3f}")

gauss_ok = False
try:
    popt_g, pcov_g = curve_fit(
        bi_gaussian, phi_labels, mass_frac,
        p0=p0_g, bounds=(lb_g, ub_g), maxfev=30000
    )
    perr_g = np.sqrt(np.diag(pcov_g))
    mu1_g, sig1_g, mu2_g, sig2_g, p_g = popt_g
    yg_fit  = bi_gaussian(phi_labels, *popt_g)
    rmse_g  = np.sqrt(np.mean((mass_frac - yg_fit) ** 2)) * 100
    BI      = (np.sqrt(2) * abs(mu2_g - mu1_g)
               / np.sqrt(sig1_g**2 + sig2_g**2)
               * np.sqrt(p_g * (1.0 - p_g)))
    gauss_ok = True
except Exception as e:
    print(f"Bi-Gaussian fit failed from clicked guess: {e}")
    print("Try re-clicking closer to each peak's true center, "
          "or widen lb_g/ub_g above.")

if gauss_ok:
    print(f"\nBi-Gaussian fit (from clicked starting guess):")
    print(f"  mu1    = {mu1_g:+.3f} Phi  +/- {perr_g[0]:.3f}")
    print(f"  sigma1 = {sig1_g:.3f} Phi  +/- {perr_g[1]:.3f}")
    print(f"  mu2    = {mu2_g:+.3f} Phi  +/- {perr_g[2]:.3f}")
    print(f"  sigma2 = {sig2_g:.3f} Phi  +/- {perr_g[3]:.3f}")
    print(f"  p      = {p_g:.4f}       +/- {perr_g[4]:.4f}")
    print(f"  BI     = {BI:.3f}  ({'bimodal' if BI > 1.1 else 'unimodal'})")
    print(f"  RMSE   = {rmse_g:.4f} wt.%")

In [ ]:
# =================================================================
# CELL 5 — plot the fit result
# =================================================================
if not gauss_ok:
    raise RuntimeError("Cannot plot: the fit in Cell 3 did not converge.")

phi_smooth = np.linspace(phi_labels.min() - 1.0, phi_labels.max() + 1.0, 1000)
CDATA, CFIT, CPOP1, CPOP2 = '#c0392b', '#2c3e50', '#2980b9', '#27ae60'

fig2 = plt.figure(figsize=(7, 5.4))
fig2.suptitle(f"{DATASET_LABEL} (click-picked starting guess)",
              fontsize=12, y=1.02, fontweight='bold')
ax2 = fig2.add_subplot(111)

ax2.bar(phi_labels, mass_pct, width=0.90, color=CDATA, alpha=0.55,
        edgecolor='grey', linewidth=0.5, label='Data')

yg_tot = bi_gaussian(phi_smooth, *popt_g) * 100
yg_c   = p_g      / (sig1_g * np.sqrt(2 * np.pi)) \
         * np.exp(-0.5 * ((phi_smooth - mu1_g) / sig1_g) ** 2) * 100
yg_f   = (1 - p_g) / (sig2_g * np.sqrt(2 * np.pi)) \
         * np.exp(-0.5 * ((phi_smooth - mu2_g) / sig2_g) ** 2) * 100

ax2.plot(phi_smooth, yg_tot, color=CFIT, lw=2.0, label='Bi-Gaussian fit')
ax2.plot(phi_smooth, yg_c, color=CPOP1, lw=1.2, ls='--',
          label=f'Pop. 1  μ₁ = {mu1_g:.2f} Φ')
ax2.plot(phi_smooth, yg_f, color=CPOP2, lw=1.2, ls='--',
          label=f'Pop. 2  μ₂ = {mu2_g:.2f} Φ')

ax2.set_xlabel('Φ', fontsize=11)
ax2.set_ylabel('Mass (wt.%)', fontsize=11)
ax2.set_title('Bi-Gaussian', fontsize=11, fontweight='bold')
ax2.set_xlim(phi_labels.min() - 1, phi_labels.max() + 1)
ax2.set_ylim(bottom=0)
ax2.legend(fontsize=8)
ax2.tick_params(labelsize=9)
plt.tight_layout()
plt.show()

# --- Parameter report ---
SEP = "=" * 62
print(f"\n{SEP}")
print(f"  TGSD FIT RESULTS (click-picked starting guess)")
print(f"  {DATASET_LABEL}")
print(f"{SEP}")
print("\n  BI-GAUSSIAN")
print(f"    mu1    = {mu1_g:+8.3f} Phi   +/- {perr_g[0]:.3f}")
print(f"    sigma1 = {sig1_g:8.3f} Phi   +/- {perr_g[1]:.3f}")
print(f"    mu2    = {mu2_g:+8.3f} Phi   +/- {perr_g[2]:.3f}")
print(f"    sigma2 = {sig2_g:8.3f} Phi   +/- {perr_g[3]:.3f}")
print(f"    p      = {p_g:8.4f}       +/- {perr_g[4]:.4f}")
print(f"    BI     = {BI:8.3f}       ({'bimodal' if BI > 1.1 else 'unimodal'})")
print(f"    RMSE   = {rmse_g:.4f} wt.%")